# 🛒 Supermarket Sales Analysis
## Retail Domain | Data Analysis Portfolio — Project 1
**Author:** Binal Doshi | MSc AI & Data Science, University of Mumbai  
**Period Covered:** January 2023 – December 2023  
**Dataset:** 2,000 transactions across 3 branches

---
### 📋 Table of Contents
1. [Setup & Data Loading](#setup)
2. [Exploratory Data Analysis (EDA)](#eda)
3. [Revenue & Sales Trends](#revenue)
4. [Product Line Analysis](#product)
5. [Branch Performance](#branch)
6. [Temporal Patterns (Day/Hour)](#temporal)
7. [Customer Behavior](#customer)
8. [Payment Method Analysis](#payment)
9. [Statistical Analysis & Correlations](#stats)
10. [Business Insights & Recommendations](#insights)


## 1. 🔧 Setup & Imports <a id='setup'></a>

In [ ]:
# Standard data analysis imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ─── Color Palette ─────────────────────────────────────────────
COLORS = {
    'primary':   '#2E4057',
    'secondary': '#048A81',
    'accent1':   '#54C6EB',
    'accent2':   '#EF946C',
    'accent3':   '#C4A35A',
    'accent4':   '#8BC34A',
    'bg':        '#F8F9FA',
    'text':      '#2C3E50',
    'grid':      '#E0E0E0',
}
PALETTE = list(COLORS.values())[:6]

plt.rcParams.update({
    'figure.facecolor': COLORS['bg'],
    'axes.facecolor':   COLORS['bg'],
    'axes.grid':        True,
    'grid.color':       COLORS['grid'],
    'grid.linewidth':   0.6,
    'font.size':        10,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})
print("✅ Libraries loaded successfully")
print(f"pandas: {pd.__version__} | numpy: {np.__version__}")


## 2. 📂 Data Loading & Initial Exploration <a id='eda'></a>

In [ ]:
# Load dataset
df = pd.read_csv('../data/supermarket_sales.csv')

# Feature Engineering
df['Date']       = pd.to_datetime(df['Date'])
df['Month']      = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['DayName']    = df['Date'].dt.day_name()
df['Hour']       = pd.to_datetime(df['Time'], format='%H:%M').dt.hour
df['Week']       = df['Date'].dt.isocalendar().week.astype(int)
df['Gross_Margin']= df['Total'] - df['Tax']

branch_map = {'A': 'Yangon (A)', 'B': 'Mandalay (B)', 'C': 'Naypyitaw (C)'}
df['Branch_Name'] = df['Branch'].map(branch_map)

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Shape:         {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date Range:    {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Missing Values:{df.isnull().sum().sum()}")
print(f"Branches:      {', '.join(df['Branch_Name'].unique())}")
print(f"Product Lines: {df['Product_Line'].nunique()} categories")
print(f"Payment Types: {', '.join(df['Payment'].unique())}")
df.head()


In [ ]:
# Data types and info
print("Column Data Types:")
print("-" * 35)
for col, dtype in df.dtypes.items():
    print(f"  {col:<22} {dtype}")


In [ ]:
# Descriptive statistics
print("\nDescriptive Statistics (Numerical Columns):")
df[['Unit_Price','Quantity','Tax','Total','Rating','Gross_Margin']].describe().round(2)


## 3. 📈 Revenue & Sales Trends <a id='revenue'></a>

In [ ]:
# Monthly Revenue Trend
monthly = df.groupby(['Month','Month_Name'])['Total'].sum().reset_index().sort_values('Month')

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(monthly['Month_Name'], monthly['Total'], alpha=0.18, color=COLORS['secondary'])
ax.plot(monthly['Month_Name'], monthly['Total'], color=COLORS['secondary'],
        linewidth=2.5, marker='o', markersize=7, markerfacecolor='white',
        markeredgecolor=COLORS['secondary'], markeredgewidth=2)
for _, row in monthly.iterrows():
    ax.annotate(f"${row['Total']:,.0f}", xy=(row['Month_Name'], row['Total']),
                xytext=(0, 10), textcoords='offset points', ha='center',
                fontsize=8, color=COLORS['primary'], fontweight='bold')
ax.set_title('Monthly Revenue Trend (2023)')
ax.set_xlabel('Month'); ax.set_ylabel('Total Revenue ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout(); plt.show()

print("\nMonthly Revenue Summary:")
print(monthly[['Month_Name','Total']].rename(columns={'Month_Name':'Month','Total':'Revenue'})
      .assign(Revenue=lambda x: x['Revenue'].map('${:,.2f}'.format)).to_string(index=False))


## 4. 🏪 Product Line Performance <a id='product'></a>

In [ ]:
# Product Line Revenue
product_rev = df.groupby('Product_Line')['Total'].sum().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Horizontal bar
bars = axes[0].barh(product_rev.index, product_rev.values,
               color=PALETTE[:len(product_rev)], edgecolor='white', height=0.6)
for bar, val in zip(bars, product_rev.values):
    axes[0].text(val + 300, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9, color=COLORS['text'])
axes[0].set_title('Revenue by Product Line')
axes[0].set_xlabel('Total Revenue ($)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_xlim(0, product_rev.max() * 1.20)

# Pie chart
total = product_rev.sum()
wedges, texts, autotexts = axes[1].pie(
    product_rev.values, labels=product_rev.index,
    autopct='%1.1f%%', colors=PALETTE[:len(product_rev)],
    startangle=90, pctdistance=0.78, wedgeprops=dict(width=0.6))
for at in autotexts: at.set_fontsize(9); at.set_fontweight('bold')
axes[1].set_title('Revenue Share by Product Line')
plt.tight_layout(); plt.show()

print("\nProduct Line Performance:")
product_stats = df.groupby('Product_Line').agg(
    Revenue=('Total','sum'), Transactions=('Invoice_ID','count'),
    Avg_Quantity=('Quantity','mean'), Avg_Rating=('Rating','mean')
).round(2)
product_stats['Revenue_Share_%'] = (product_stats['Revenue'] / product_stats['Revenue'].sum() * 100).round(1)
product_stats['Revenue'] = product_stats['Revenue'].map('${:,.0f}'.format)
print(product_stats.to_string())


## 5. 🏬 Branch Performance Analysis <a id='branch'></a>

In [ ]:
# Branch Comparison
branch_stats = df.groupby('Branch_Name').agg(
    Revenue=('Total','sum'), Transactions=('Invoice_ID','count'),
    Avg_Rating=('Rating','mean'), Avg_Spend=('Total','mean')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = [('Revenue', '$', ',.0f'), ('Transactions', '', ',d'), ('Avg_Rating', '', '.2f')]
branch_colors = [COLORS['primary'], COLORS['secondary'], COLORS['accent2']]

for ax, (col, prefix, fmt) in zip(axes, metrics):
    bars = ax.bar(branch_stats['Branch_Name'], branch_stats[col],
                  color=branch_colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars, branch_stats[col]):
        label = f'{prefix}{val:{fmt}}'
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + branch_stats[col].max()*0.01,
                label, ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(col.replace('_',' '))
    ax.tick_params(axis='x', labelsize=8)
    if col == 'Revenue':
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
fig.suptitle('Branch Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print(branch_stats.to_string(index=False))


## 6. 📅 Temporal Sales Patterns <a id='temporal'></a>

In [ ]:
# Day of Week & Hourly Heatmap
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_sales = df.groupby('DayName')['Total'].sum().reindex(day_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Day bar chart
colors_day = [COLORS['accent2'] if v == day_sales.max() else COLORS['primary'] for v in day_sales]
bars = axes[0].bar(day_sales.index, day_sales.values, color=colors_day, edgecolor='white', width=0.6)
for bar, val in zip(bars, day_sales.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=8)
axes[0].set_title('Revenue by Day of Week')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=25)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
best_patch = mpatches.Patch(color=COLORS['accent2'], label='Peak Day')
axes[0].legend(handles=[best_patch])

# Hourly line
hourly = df.groupby('Hour')['Total'].sum()
axes[1].fill_between(hourly.index, hourly.values, alpha=0.2, color=COLORS['primary'])
axes[1].plot(hourly.index, hourly.values, color=COLORS['primary'],
             linewidth=2, marker='o', markersize=5)
axes[1].set_title('Revenue by Hour of Day')
axes[1].set_xlabel('Hour'); axes[1].set_ylabel('Revenue ($)')
axes[1].set_xticks(range(hourly.index.min(), hourly.index.max()+1))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout(); plt.show()

print(f"Peak Day:  {day_sales.idxmax()} (${day_sales.max():,.0f})")
print(f"Peak Hour: {hourly.idxmax()}:00 (${hourly.max():,.0f})")


In [ ]:
# Heatmap
hour_day = df.groupby(['DayName', 'Hour'])['Total'].sum().unstack(fill_value=0)
hour_day = hour_day.reindex(day_order)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(hour_day, cmap='YlOrRd', ax=ax, linewidths=0.3,
            cbar_kws={'label': 'Revenue ($)', 'shrink': 0.8})
ax.set_title('Sales Heatmap: Day of Week vs Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Day of Week')
plt.tight_layout(); plt.show()


## 7. 👥 Customer Behavior Analysis <a id='customer'></a>

In [ ]:
# Gender & Customer Type Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender × Product Line
gender_product = df.groupby(['Product_Line','Gender'])['Total'].sum().unstack()
gender_product.plot(kind='bar', ax=axes[0],
                    color=[COLORS['accent2'], COLORS['secondary']], edgecolor='white', width=0.7)
axes[0].set_title('Product Line Preference by Gender')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].legend(title='Gender')

# Member vs Normal
ctype = df.groupby('Customer_Type').agg(Avg_Spend=('Total','mean'), Count=('Invoice_ID','count')).reset_index()
x = np.arange(len(ctype))
w = 0.35
axes[1].bar(x - w/2, ctype['Avg_Spend'], width=w, label='Avg Spend', color=COLORS['primary'])
ax2 = axes[1].twinx()
ax2.bar(x + w/2, ctype['Count'], width=w, label='Transactions', color=COLORS['accent1'], alpha=0.8)
axes[1].set_title('Member vs Normal Customer')
axes[1].set_xticks(x); axes[1].set_xticklabels(ctype['Customer_Type'])
axes[1].set_ylabel('Avg Transaction ($)')
ax2.set_ylabel('Transactions', color=COLORS['accent1'])
lines1,lbl1 = axes[1].get_legend_handles_labels()
lines2,lbl2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1+lines2, lbl1+lbl2, fontsize=8)
plt.tight_layout(); plt.show()

print("Customer Type Summary:")
print(df.groupby('Customer_Type').agg(
    Avg_Spend=('Total','mean'), Total_Revenue=('Total','sum'),
    Transactions=('Invoice_ID','count')).round(2).to_string())


## 8. 💳 Payment Method Analysis <a id='payment'></a>

In [ ]:
# Payment Analysis
pay_counts = df['Payment'].value_counts()
pay_rev = df.groupby('Payment')['Total'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
wedges, texts, autotexts = axes[0].pie(
    pay_counts.values, labels=pay_counts.index, autopct='%1.1f%%',
    colors=[COLORS['primary'], COLORS['secondary'], COLORS['accent2']],
    startangle=90, pctdistance=0.75, wedgeprops=dict(width=0.6))
for at in autotexts: at.set_fontsize(10); at.set_fontweight('bold')
axes[0].set_title('Payment Method Distribution')

bars = axes[1].bar(pay_rev.index, pay_rev.values,
        color=[COLORS['primary'], COLORS['secondary'], COLORS['accent2']],
        edgecolor='white', width=0.5)
for i, (idx, val) in enumerate(pay_rev.items()):
    axes[1].text(i, val + 200, f'${val:,.0f}', ha='center', fontsize=9)
axes[1].set_title('Revenue by Payment Method')
axes[1].set_ylabel('Revenue ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout(); plt.show()

print("Payment Method Summary:")
pay_summary = df.groupby('Payment').agg(
    Transactions=('Invoice_ID','count'),
    Revenue=('Total','sum'),
    Avg_Transaction=('Total','mean')
).round(2)
pay_summary['Revenue_Share_%'] = (pay_summary['Revenue'] / pay_summary['Revenue'].sum() * 100).round(1)
print(pay_summary.to_string())


## 9. 📊 Statistical Analysis <a id='stats'></a>

In [ ]:
# Correlation & Rating Distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Rating histogram
axes[0].hist(df['Rating'], bins=20, color=COLORS['secondary'], edgecolor='white', alpha=0.85)
axes[0].axvline(df['Rating'].mean(), color=COLORS['accent2'], lw=2, ls='--',
                label=f"Mean: {df['Rating'].mean():.2f}")
axes[0].axvline(df['Rating'].median(), color=COLORS['primary'], lw=2, ls=':',
                label=f"Median: {df['Rating'].median():.2f}")
axes[0].set_title('Customer Rating Distribution')
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Frequency')
axes[0].legend()

# Correlation heatmap
corr = df[['Unit_Price','Quantity','Tax','Total','Rating','Gross_Margin']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', ax=axes[1],
            cmap='coolwarm', center=0, linewidths=0.5)
axes[1].set_title('Feature Correlation Matrix')
plt.tight_layout(); plt.show()

print(f"Rating Stats — Mean: {df['Rating'].mean():.2f} | Std: {df['Rating'].std():.2f} | Min: {df['Rating'].min()} | Max: {df['Rating'].max()}")


## 10. 💡 Business Insights & Recommendations <a id='insights'></a>

---

### Key Findings

| # | Insight | Evidence |
|---|---------|----------|
| 1 | **Health & Beauty** is the top-revenue product line | Highest total revenue |
| 2 | **Saturday** is the peak sales day | Day-of-week analysis |
| 3 | **E-wallet** leads in payment adoption | Payment distribution |
| 4 | **6 PM (18:00)** is the busiest shopping hour | Hourly analysis |
| 5 | Customer satisfaction is **independent** of spend amount | Near-zero Rating correlation |
| 6 | **All 3 branches** perform comparably | Branch analysis |

---

### Strategic Recommendations

1. **Stock Prioritization** — Allocate 20% more inventory to Health & Beauty
2. **Weekend Promotions** — Launch Saturday loyalty events and flash sales
3. **Digital Payment Incentives** — E-wallet cashback partnerships
4. **Evening Staff Surge** — Add staff from 5–9 PM for peak hours
5. **Membership Enhancement** — Upgrade benefits to convert Normal → Member customers
6. **Cross-Branch Learning** — Share best practices between branches


In [ ]:
# Final Summary Statistics
print("=" * 55)
print("   SUPERMARKET SALES — FINAL SUMMARY REPORT")
print("=" * 55)
print(f"  Total Revenue       : ${df['Total'].sum():>12,.2f}")
print(f"  Total Transactions  : {len(df):>12,}")
print(f"  Avg Transaction     : ${df['Total'].mean():>12.2f}")
print(f"  Avg Customer Rating : {df['Rating'].mean():>12.2f} / 10")
print(f"  Best Product Line   : {df.groupby('Product_Line')['Total'].sum().idxmax():>20}")
print(f"  Best Branch         : {df.groupby('Branch_Name')['Total'].sum().idxmax():>20}")
print(f"  Peak Sales Day      : {df.groupby('DayName')['Total'].sum().idxmax():>20}")
print(f"  Peak Hour           : {df.groupby('Hour')['Total'].sum().idxmax():>19}:00")
print(f"  Top Payment Method  : {df.groupby('Payment')['Total'].sum().idxmax():>20}")
print("=" * 55)
print("  Analysis by: Binal Doshi | MSc AI & Data Science")
print("  University of Mumbai | Portfolio Project 1")
print("=" * 55)
